# Eval 1: Initial Setup

This Eval covers environment setup, model loading, and sanity checks before benchmarking or Part 9.

Run from the top through the quick-forward and training scaffold sections in order.

### Pipeline
```
Input tokens
  -> HF Backbone  (all hidden states)
  -> LogicProjection  [B,S,H] -> [B,S,L]
  -> LogicStream  (one LogicLayer per transformer layer)
      |- RoutingModule  [B,S,L] -> [B,S,G]
      |- AND / OR / NOT gates  (hard-clip at 0.5)
      |- logic_update  [B,G] -> [B,L]
  -> FusionMLP  (residual fusion of last hidden state + logic state)
  -> Task head  [B,H] -> [B,num_labels]
```

## W&B Tracking Reference

For chart definitions used across all Evals and per-Eval tracking guidance, see:
- `logic/docs/wandb_chart_tracking.md`

Quick reminder before launching runs:
- Keep `wandb.group` consistent per Eval batch.
- Use descriptive `wandb.run_name` values.
- Add key config columns in W&B Runs (for fast filtering/comparison).

In [ ]:
# Colab/runtime dependencies for this notebook.
%pip install -q torch transformers datasets peft matplotlib wandb pyyaml huggingface_hub
%pip install -q bitsandbytes accelerate   # required for 4-bit Llama loading

In [ ]:
import os
import sys
import platform
import importlib.util
import subprocess

in_colab = importlib.util.find_spec("google.colab") is not None
print("In Google Colab:", in_colab)

runtime = "CPU"
if "COLAB_TPU_ADDR" in os.environ:
    runtime = "TPU"
elif os.path.exists("/proc/driver/nvidia/version"):
    runtime = "GPU"

print("Runtime type:", runtime)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

if runtime == "GPU":
    print("\nGPU details:")
    subprocess.run(["nvidia-smi"], check=False)

In [ ]:
# -- Colab setup for private repo clone/pull (safe token handling) ---------------
# Skip this cell if running locally in VS Code.
import os
import sys

if "google.colab" in sys.modules:
    from getpass import getpass
    from google.colab import userdata

    REPO_URL = "github.com/Ancorro/AI535project.git"
    REPO_DIR = "/content/AI535project"

    # Prefer Colab Secrets first, then env var, then prompt as last resort.
    gh_token = (
        userdata.get("GITHUB_TOKEN")
        or userdata.get("GH_TOKEN")
        or os.getenv("GITHUB_TOKEN")
    )
    if not gh_token:
        gh_token = getpass("GitHub PAT (repo read access): ")

    auth_url = f"https://{gh_token}@{REPO_URL}"
    if os.path.isdir(REPO_DIR):
        print("Repo exists, pulling latest changes...")
        ! git -C {REPO_DIR} pull
    else:
        print("Repo not found, cloning...")
        ! git clone {auth_url} {REPO_DIR}

    del gh_token, auth_url

    %cd /content/AI535project/logic

In [ ]:
import os
import sys
from pathlib import Path

# Make the top-level project importable from repo root, logic/, or logic/notebooks/.
def _find_repo_root(start: Path) -> Path:
    required = [
        Path("logic/core/logic_llama_model.py"),
        Path("scripts/test_logic_vs_llama31.py"),
        Path("train/train.py"),
    ]
    for candidate in [start, *start.parents]:
        if all((candidate / rel).exists() for rel in required):
            return candidate

    # Common Colab clone locations.
    for candidate in [Path("/content/AI535project"), Path("/content/drive/MyDrive/AI535project")]:
        if all((candidate / rel).exists() for rel in required):
            return candidate

    raise FileNotFoundError(
        "Could not locate AI535project repo root. Start from the repo root or set REPO_ROOT manually."
    )

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

from logic.core.logic_llama_model import LogicLlamaModel, LogicModelOutput
from logic.core.logic_gates import and_gate, or_gate, not_gate, compose_gate_outputs

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1  Load a Pre-Trained HuggingFace Backbone

Primary path shown below:
- **Option B** — Llama 3.1 8B (recommended for the full logic architecture)

Pooling behavior is selected automatically for decoder-style models:
- Decoder (Llama-style) -> pool from **last non-padding token** (correct for causal LMs)

In [ ]:
# Option A removed intentionally to keep this notebook Llama-only.
# Use Option B below.

### Access Preflight (run before loading Llama)

This check validates HuggingFace authentication/access and dataset availability before any large model weights are downloaded.

In [ ]:
# Lightweight preflight checks for gated model access and dataset availability.
# Run this before Option B or benchmark cells that load Llama weights.

import os
import sys
from datasets import load_dataset_builder
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

MODEL_CANDIDATES = [
    "meta-llama/Meta-Llama-3.1-8B",            # used in Option B
    "meta-llama/Llama-3.1-8B-Instruct",        # used in benchmark
]
DATASET_CHECKS = [
    ("glue", "cola"),
    ("longface/ProofWriter", None),
]

# Resolve token from Colab secrets first, then env vars.
hf_token = None
if "google.colab" in sys.modules:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")

print("== Llama Access Preflight ==")
print("Token found:", bool(hf_token))

api = HfApi()
model_ok = {}
for model_id in MODEL_CANDIDATES:
    try:
        _ = api.model_info(repo_id=model_id, token=hf_token)
        model_ok[model_id] = True
        print(f"[OK] model access: {model_id}")
    except HfHubHTTPError as e:
        model_ok[model_id] = False
        print(f"[FAIL] model access: {model_id} -> HTTP {getattr(e.response, 'status_code', 'unknown')}")
    except Exception as e:
        model_ok[model_id] = False
        print(f"[FAIL] model access: {model_id} -> {type(e).__name__}: {e}")

dataset_ok = {}
for ds_name, ds_cfg in DATASET_CHECKS:
    try:
        _ = load_dataset_builder(ds_name, ds_cfg) if ds_cfg else load_dataset_builder(ds_name)
        dataset_ok[(ds_name, ds_cfg)] = True
        suffix = f"/{ds_cfg}" if ds_cfg else ""
        print(f"[OK] dataset access: {ds_name}{suffix}")
    except Exception as e:
        dataset_ok[(ds_name, ds_cfg)] = False
        suffix = f"/{ds_cfg}" if ds_cfg else ""
        print(f"[FAIL] dataset access: {ds_name}{suffix} -> {type(e).__name__}: {e}")

LLAMA_PREFLIGHT_OK = all(model_ok.values()) and all(dataset_ok.values())
print("\nLLAMA_PREFLIGHT_OK =", LLAMA_PREFLIGHT_OK)
if not LLAMA_PREFLIGHT_OK:
    print("Resolve failed checks before loading Llama to avoid slow failed downloads.")

In [ ]:
# -- Option B: Llama 3.1 8B -- fp16, full precision ----------------------------
# Requires HuggingFace access grant for meta-llama/Meta-Llama-3.1-8B
# In Colab, store HF token in Secrets as HF_TOKEN (or HUGGINGFACE_TOKEN).
# Local fallback uses environment variables or existing `huggingface-cli login`.

# Hard guard: do not start large model download unless preflight passed.
if "LLAMA_PREFLIGHT_OK" not in globals():
    raise RuntimeError(
        "Preflight not found. Run the 'Access Preflight' cell first, then rerun this cell."
    )
if not LLAMA_PREFLIGHT_OK:
    raise RuntimeError(
        "LLAMA_PREFLIGHT_OK is False. Fix token/access or dataset checks before loading Llama."
    )

MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B"

hf_token = None
if "google.colab" in sys.modules:
    # Prefer Colab userdata/Secrets when running in browser Colab.
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN")
else:
    hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")

# Make token visible to downstream script imports that call from_pretrained directly.
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_TOKEN"] = hf_token

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=hf_token if hf_token else None,
)
tokenizer.pad_token = tokenizer.eos_token   # Llama has no dedicated pad token

backbone = AutoModel.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True,
    token=hf_token if hf_token else None,
)
backbone.config.pad_token_id = tokenizer.eos_token_id

print(f"Hidden size  : {backbone.config.hidden_size}")
print(f"Num layers   : {backbone.config.num_hidden_layers}")

# Scale logic dims up to match the larger hidden space
LOGIC_DIM, NUM_GATES, FUSION_HIDDEN = 256, 32, 1024

## 2  Wrap with LogicLlamaModel

In [ ]:
# LOGIC_DIM / NUM_GATES / FUSION_HIDDEN are set by Option A or B above.
# Defaults here in case this cell is run standalone.
LOGIC_DIM     = locals().get("LOGIC_DIM", 128)
NUM_GATES     = locals().get("NUM_GATES", 16)
FUSION_HIDDEN = locals().get("FUSION_HIDDEN", 512)

ROUTING_TOP_K = 2
NUM_LABELS    = 2

model = LogicLlamaModel(
    backbone=backbone,
    stream_logic_during_backbone=True,
    logic_dim=LOGIC_DIM,
    num_gates=NUM_GATES,
    routing_top_k=ROUTING_TOP_K,
    fusion_hidden_dim=FUSION_HIDDEN,
    num_labels=NUM_LABELS,
)

# Match logic modules to backbone device + dtype to avoid Half/Float mismatch.
runtime_device = next(backbone.parameters()).device
runtime_dtype  = next(backbone.parameters()).dtype

if not hasattr(backbone, "hf_device_map"):
    model = model.to(device=runtime_device, dtype=runtime_dtype)
else:
    model.logic_projection.to(device=runtime_device, dtype=runtime_dtype)
    model.logic_stream.to(device=runtime_device, dtype=runtime_dtype)
    model.fusion.to(device=runtime_device, dtype=runtime_dtype)
    model.task_head.to(device=runtime_device, dtype=runtime_dtype)
    print(f"Llama backbone spread across: {backbone.hf_device_map}")

print(f"Runtime device : {runtime_device}")
print(f"Runtime dtype  : {runtime_dtype}")
print(f"Causal pooling : {model._causal}")
total_params  = sum(p.numel() for p in model.parameters())
logic_params  = sum(p.numel() for m in [
    model.logic_projection, model.logic_stream, model.fusion, model.task_head
] for p in m.parameters())
print(f"Total params      : {total_params:,}")
print(f"Logic/head params : {logic_params:,}  ({100*logic_params/total_params:.1f}%)")

## 3  Quick Forward Pass

In [ ]:
sentences = [
    "All cats are animals. Felix is a cat.",
    "If it rains the ground is wet. The ground is not wet.",
]

enc = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True, max_length=64)
input_ids      = enc["input_ids"].to(runtime_device)
attention_mask = enc["attention_mask"].to(runtime_device)

model.eval()
with torch.no_grad():
    out: LogicModelOutput = model(input_ids, attention_mask)

print("logits shape       :", out.logits.shape)
print("fused_hidden shape :", out.fused_hidden.shape)
print("routing layers     :", len(out.routing_history))
print("logits:\n", out.logits)

## 4  Training Setup

> **LoRA is disabled.** The Stage 2 cell below is fully commented out. To re-enable, uncomment it and run `pip install peft`.

### Stage 1 – freeze backbone, train only logic + head

In [ ]:
# ---------- Stage 1: freeze backbone ------------------------------------------
model.freeze_backbone()

trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print(f"{len(trainable)} trainable parameter tensors (logic + head only)")

# FP16 logic/head training is sensitive; use a smaller LR and stable AdamW eps.
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=1e-2,
    eps=1e-6,
)
criterion = nn.CrossEntropyLoss()

In [ ]:
# ---------- Stage 2 (optional): LoRA on the backbone --------------------------
# Uncomment after `pip install peft`

# model.enable_lora(r=8, lora_alpha=16, dropout=0.05)
# optimizer = torch.optim.AdamW(
#     [p for p in model.parameters() if p.requires_grad], lr=2e-4
# )

In [ ]:
# ---------- Toy training loop (replace with your DataLoader) ------------------
DUMMY_LABELS = torch.tensor([1, 0], device=runtime_device, dtype=torch.long)

model.train()
for step in range(10):
    optimizer.zero_grad(set_to_none=True)
    out = model(input_ids, attention_mask)
    loss = criterion(out.logits.float(), DUMMY_LABELS)

    # Stop early if values became invalid so we can diagnose instead of propagating NaNs.
    if not torch.isfinite(loss):
        print(f"step {step:02d}  loss is non-finite: {loss.item()}")
        print("logits min/max:", out.logits.min().item(), out.logits.max().item())
        break

    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad],
        max_norm=1.0,
    )
    optimizer.step()

    if step % 2 == 0:
        print(f"step {step:02d}  loss={loss.item():.4f}")

print("Done.")

## 5  Inspect Gate Activations

Show the hard-clipped binary output of the AND / OR / NOT gates for the first batch item.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 110

model.eval()
with torch.no_grad():
    out = model(input_ids, attention_mask)

# Access gate values from the first logic layer internals
# Re-run a single layer manually so we can intercept gate_values
from logic.core.logic_gates import _clip, compose_gate_outputs
from logic.core.routing import RoutingModule

# Pull the first example's projected states out
with torch.no_grad():
    bb_out = model.backbone(input_ids[:1], attention_mask[:1])
    layer_hidden = list(bb_out.hidden_states[1:])          # per-layer [1,S,H]
    proj_states  = [model.logic_projection(h) for h in layer_hidden]

    first_layer   = model.logic_stream.layers[0]
    token_logic   = proj_states[0]                         # [1,S,L]
    routing_w     = first_layer.routing(token_logic, inference=True)   # [1,S,G]
    gate_inputs   = torch.einsum("bsg,bsl->bgl", routing_w, token_logic)  # [1,G,L]
    gate_values   = torch.sigmoid(first_layer.gate_scalar(gate_inputs).squeeze(-1))  # [1,G]

    paired     = torch.roll(gate_values, shifts=1, dims=-1)
    and_out    = and_gate(gate_values, paired)   # binary [1,G]
    or_out     = or_gate(gate_values,  paired)   # binary [1,G]
    not_out    = not_gate(gate_values)            # binary [1,G]

fig, axes = plt.subplots(1, 4, figsize=(14, 2.5))
data = [
    ("gate_values (pre-clip)", gate_values[0].cpu()),
    ("AND output",  and_out[0].cpu()),
    ("OR output",   or_out[0].cpu()),
    ("NOT output",  not_out[0].cpu()),
]
for ax, (title, vals) in zip(axes, data):
    ax.bar(range(len(vals)), vals.numpy(), color="steelblue", edgecolor="k", linewidth=0.5)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Gate index")
    ax.set_ylim(-0.05, 1.1)
    ax.axhline(0.5, color="red", linestyle="--", linewidth=0.8, label="threshold")

axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

## 6  Visualize Routing Weights Across Layers

In [ ]:
# routing_history: list of [B,S,G] tensors, one per layer
# Average over tokens, take first batch item → [num_layers, G]
with torch.no_grad():
    out = model(input_ids[:1], attention_mask[:1])

route_mat = torch.stack(
    [rw[0].mean(dim=0) for rw in out.routing_history]  # [num_layers, G]
).cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(route_mat, aspect="auto", cmap="viridis", interpolation="nearest")
ax.set_xlabel("Gate index")
ax.set_ylabel("Transformer layer")
ax.set_title("Mean token-to-gate routing weights (first example)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 7  Save / Load Checkpoint

In [ ]:
CKPT_PATH = "logic_model.pt"

# --- Save ---
torch.save(model.state_dict(), CKPT_PATH)
print(f"Saved checkpoint to {CKPT_PATH}")

# --- Reload ---
model_fresh = LogicLlamaModel(
    model_name=MODEL_NAME,
    stream_logic_during_backbone=True,
    logic_dim=LOGIC_DIM,
    num_gates=NUM_GATES,
    routing_top_k=ROUTING_TOP_K,
    fusion_hidden_dim=FUSION_HIDDEN,
    num_labels=NUM_LABELS,
).to(device)
model_fresh.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model_fresh.eval()
print("Checkpoint loaded successfully.")

## 8  HyperSmooth Setup Scaffold

These cells set up a future hyperparameter smoother loop using `HyperSmooth` and `make_quick_objective` from `hyper/gSmooth_done.py`.

They are intentionally lightweight placeholders so the notebook stays runnable now.

In [ ]:
# HyperSmooth imports + hyperparameter encoding template
from hyper.gSmooth_done import HyperSmooth, make_quick_objective
import numpy as np

# Example: x = [log10(lr), log10(weight_decay)]
def decode_x(x: np.ndarray) -> dict:
    x = np.asarray(x, dtype=float).reshape(-1)
    return {
        "lr": float(10.0 ** x[0]),
        "weight_decay": float(10.0 ** x[1]),
    }

def model_factory():
    """Build a fresh logic wrapper using the already-loaded backbone.

    For HyperSmooth speed, this version freezes the backbone and trains only
    logic/head modules. The backbone object is reused to avoid repeated
    multi-GB model loads in the objective loop.
    """
    m = LogicLlamaModel(
        backbone=backbone,
        stream_logic_during_backbone=True,
        logic_dim=LOGIC_DIM,
        num_gates=NUM_GATES,
        routing_top_k=ROUTING_TOP_K,
        fusion_hidden_dim=FUSION_HIDDEN,
        num_labels=NUM_LABELS,
    )
    m.freeze_backbone()
    m.logic_projection.to(device=runtime_device, dtype=runtime_dtype)
    m.logic_stream.to(device=runtime_device, dtype=runtime_dtype)
    m.fusion.to(device=runtime_device, dtype=runtime_dtype)
    m.task_head.to(device=runtime_device, dtype=runtime_dtype)
    return m

def optimizer_factory(model, hparams: dict):
    # Only train logic/head params for fast objective evaluations.
    params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(
        params,
        lr=hparams["lr"],
        weight_decay=hparams["weight_decay"],
    )

# Suggested starting point for smoother search space
x0 = np.array([-3.0, -4.0], dtype=float)  # lr=1e-3, wd=1e-4

In [ ]:
# HyperSmooth objective + run template (safe scaffold)

# TODO: provide DataLoaders; loss is set here for convenience
train_loader = None
val_loader = None
loss_fn = nn.CrossEntropyLoss()

def build_hypersmooth_objective():
    if train_loader is None or val_loader is None:
        raise RuntimeError(
            "Set train_loader and val_loader before building M(x)."
        )
    return make_quick_objective(
        model_factory=model_factory,
        optimizer_factory=optimizer_factory,
        decode_x=decode_x,
        train_loader=train_loader,
        val_loader=val_loader,
        loss_fn=loss_fn,
        device=str(runtime_device),
        train_steps=40,
        val_steps=8,
        seed=42,
        num_seeds=1,
    )

# When ready to run, uncomment:
# M = build_hypersmooth_objective()
# hs = HyperSmooth()
# x_best = hs.run(
#     x=x0, n=8, M=M, B=0.9, lr=0.05, eta=0.05, steps=60,
#     antithetic=True, grad_clip=10.0,
# )
# print("x_best:", x_best)
# print("decoded:", decode_x(x_best))
# print("M(x_best):", M(x_best))

# Eval 2: First Test (Single Benchmark)

Run this eval to do the first end-to-end benchmark of one project run against Llama 3.1 8B.

This section is the quick validation pass before running Part 9 four-way experiments.

If your run directory is different, only change `RUN_DIR` in the next code cell.

In [ ]:
from pathlib import Path
import json
import os
import sys

# Resolve repo root in local VS Code or remote Colab layouts.
repo_root = Path.cwd()
script_rel = Path("scripts") / "test_logic_vs_llama31.py"

if not (repo_root / script_rel).exists():
    for candidate in [repo_root, *repo_root.parents]:
        if (candidate / script_rel).exists():
            repo_root = candidate
            break

if not (repo_root / script_rel).exists():
    common_roots = [
        Path("/content/AI535project"),
        Path("/content/drive/MyDrive/AI535project"),
    ]
    for candidate in common_roots:
        if (candidate / script_rel).exists():
            repo_root = candidate
            break

if not (repo_root / script_rel).exists():
    raise FileNotFoundError(
        "Could not locate scripts/test_logic_vs_llama31.py. "
        "Set repo_root manually to your cloned AI535project path."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.test_logic_vs_llama31 import run_benchmark, run_benchmark_with_model

# ---------------- Benchmark controls ----------------
# Keep this False for stable Colab compare runs when your notebook model is itself Llama-8B.
USE_NOTEBOOK_MODEL = False
RUN_DIR = "runs/run_10228"
DATASETS = ["cola", "proofwriter"]
SPLIT = "validation"
MAX_SAMPLES = 64
BATCH_SIZE = 8
MAX_LENGTH = 256
LLAMA_MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
SKIP_LLAMA = False
OUTPUT_PATH = "runs/logic_vs_llama31_report.json"

# If Option B already fetched a token, expose it for script-level from_pretrained calls.
if "hf_token" in globals() and hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_TOKEN"] = hf_token

has_notebook_model = "model" in globals() and "tokenizer" in globals()
model_name_for_report = None
model_looks_like_llama = False
if has_notebook_model:
    model_name_for_report = str(globals().get("MODEL_NAME", model.__class__.__name__))
    model_looks_like_llama = "llama" in model_name_for_report.lower()

# Avoid trying to hold two 8B Llama models at once in Colab (common OOM).
if USE_NOTEBOOK_MODEL and has_notebook_model and model_looks_like_llama and not SKIP_LLAMA:
    print(
        "Detected notebook project model is Llama-like; switching to RUN_DIR project model "
        "to avoid loading two 8B models at once."
    )
    USE_NOTEBOOK_MODEL = False

if USE_NOTEBOOK_MODEL and has_notebook_model:
    report = run_benchmark_with_model(
        project_model=model,
        project_tokenizer=tokenizer,
        datasets=DATASETS,
        split=SPLIT,
        max_samples=MAX_SAMPLES,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        llama_model_name=LLAMA_MODEL_NAME,
        skip_llama=SKIP_LLAMA,
        output=OUTPUT_PATH,
        project_model_name=model_name_for_report,
    )
    print("Used in-notebook model/tokenizer for benchmark.")
else:
    report = run_benchmark(
        run_dir=RUN_DIR,
        datasets=DATASETS,
        split=SPLIT,
        max_samples=MAX_SAMPLES,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        llama_model_name=LLAMA_MODEL_NAME,
        skip_llama=SKIP_LLAMA,
        output=OUTPUT_PATH,
    )
    print("Used RUN_DIR project model for benchmark:", RUN_DIR)

print(json.dumps(report, indent=2)[:4000])
print("Report written to", repo_root / OUTPUT_PATH)

# Eval 3: Part 9 (Four-Way Variants + Visuals)

This eval has two subparts:
- 9A: train + benchmark four controlled variants
- 9B: plot visual diagnostics from 9A outputs

Run these 4 code cells in order for 9A:
1. Setup paths and base config: `#VSC-e2fedbbe`
2. Build and write variant configs: `#VSC-5eb4d844`
3. Train all four variants: `#VSC-d906de2e`
4. Benchmark all variants and write summary JSON: `#VSC-469ad0e7`

Variant definitions:
- Baseline A: frozen backbone + classifier head
- Baseline B: frozen backbone + LoRA + classifier head
- Logic A: frozen backbone + logic modules + classifier head
- Logic B: frozen backbone + LoRA + logic modules + classifier head

Output to expect after step 4:
- `runs/four_way_comparison_summary.json`
- `runs/logic_vs_llama31_<variant>.json`

Then run 9B plotting cells below (`#VSC-026e6384`, optional `#VSC-f8ba9ea3`).

In [ ]:
from pathlib import Path
import copy
import json
import os
import subprocess
import sys

# Resolve repo root from local VS Code or Colab paths.
repo_root = Path.cwd()
if not (repo_root / "train" / "train.py").exists():
    for candidate in [repo_root, *repo_root.parents]:
        if (candidate / "train" / "train.py").exists():
            repo_root = candidate
            break

if not (repo_root / "train" / "train.py").exists():
    for candidate in [Path("/content/AI535project"), Path("/content/drive/MyDrive/AI535project")]:
        if (candidate / "train" / "train.py").exists():
            repo_root = candidate
            break

if not (repo_root / "train" / "train.py").exists():
    raise FileNotFoundError("Could not locate train/train.py from current working directory.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from train.train import _load_config

BASE_CONFIG_PATH = repo_root / "configs" / "config.yaml"
EXPERIMENT_CONFIG_DIR = repo_root / "configs" / "experiments"
EXPERIMENT_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# Use notebook model name when available; else use config model name.
base_cfg = _load_config(BASE_CONFIG_PATH)
base_model_name = globals().get("MODEL_NAME", base_cfg.get("model", {}).get("model_name", "meta-llama/Meta-Llama-3.1-8B"))

# Configure wandb using Colab secret or environment variables.
wandb_cfg = base_cfg.setdefault("wandb", {})
wandb_enabled = bool(wandb_cfg.get("enabled", False))

if wandb_enabled:
    wandb_key = None

    # Prefer Colab secrets when running in Colab.
    if "google.colab" in sys.modules:
        try:
            from google.colab import userdata
            wandb_key = userdata.get("WANDB_API_KEY") or userdata.get("WANDB_KEY")
        except Exception:
            wandb_key = None

    # Fallback to environment variables for local/CI runs.
    if not wandb_key:
        wandb_key = os.getenv("WANDB_API_KEY") or os.getenv("WANDB_KEY")

    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
        try:
            import wandb
            wandb.login(key=wandb_key, relogin=True)
            print("wandb login successful.")
        except Exception as e:
            print(f"wandb login failed ({type(e).__name__}: {e}). Disabling wandb for Part 9 variant runs.")
            wandb_cfg["enabled"] = False
    else:
        print("No WANDB key found in Colab secrets or environment; disabling wandb for Part 9 variant runs.")
        wandb_cfg["enabled"] = False
else:
    print("wandb disabled in base config.")

print("Repo root:", repo_root)
print("Base model for four-way variants:", base_model_name)

In [ ]:
from datetime import datetime

EVAL3_WANDB_GROUP = f"eval3_four_way_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print("Eval 3 wandb group:", EVAL3_WANDB_GROUP)


def _build_variant_config(base_cfg: dict, variant_name: str, model_name: str, num_labels: int = 2) -> dict:
    cfg = copy.deepcopy(base_cfg)

    cfg.setdefault("model", {})
    cfg.setdefault("train", {})
    cfg.setdefault("data", {})
    cfg.setdefault("wandb", {})

    cfg["model"]["model_name"] = model_name
    cfg["model"]["num_labels"] = num_labels
    cfg["model"]["lora"] = cfg["model"].get("lora", {}) or {}

    # Keep data/train defaults lightweight for Colab experimentation.
    cfg["data"]["dataset"] = cfg["data"].get("dataset", "glue")
    cfg["data"]["config"] = cfg["data"].get("config", "cola")
    cfg["data"]["max_samples"] = int(cfg["data"].get("max_samples", 256) or 256)

    cfg["train"]["epochs"] = int(cfg["train"].get("epochs", 3))
    cfg["train"]["batch_size"] = int(cfg["train"].get("batch_size", 8))

    # Variant-specific switches.
    if variant_name == "baseline_a":
        cfg["model"]["use_logic_stream"] = False
        cfg["model"]["freeze_backbone"] = True
        cfg["model"]["lora"] = {"enabled": False}
    elif variant_name == "baseline_b":
        cfg["model"]["use_logic_stream"] = False
        cfg["model"]["freeze_backbone"] = True
        cfg["model"]["lora"] = {
            "enabled": True,
            "r": 8,
            "lora_alpha": 16,
            "dropout": 0.05,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        }
    elif variant_name == "logic_a":
        cfg["model"]["use_logic_stream"] = True
        cfg["model"]["freeze_backbone"] = True
        cfg["model"]["lora"] = {"enabled": False}
    elif variant_name == "logic_b":
        cfg["model"]["use_logic_stream"] = True
        cfg["model"]["freeze_backbone"] = True
        cfg["model"]["lora"] = {
            "enabled": True,
            "r": 8,
            "lora_alpha": 16,
            "dropout": 0.05,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        }
    else:
        raise ValueError(f"Unknown variant: {variant_name}")

    # Per-eval wandb grouping and per-test run naming.
    cfg["wandb"]["group"] = EVAL3_WANDB_GROUP
    cfg["wandb"]["run_name"] = f"eval3_{variant_name}"

    return cfg


VARIANTS = ["baseline_a", "baseline_b", "logic_a", "logic_b"]
variant_config_paths = {}
for name in VARIANTS:
    cfg = _build_variant_config(base_cfg, name, model_name=base_model_name)
    out_path = EXPERIMENT_CONFIG_DIR / f"{name}.yaml"
    import yaml
    with open(out_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    variant_config_paths[name] = out_path

print("Wrote variant configs:")
for k, v in variant_config_paths.items():
    print(f"  - {k}: {v}")

In [ ]:
# Train each variant. This may take a while on 8B backbones.
# Tip: reduce epochs/max_samples in configs/experiments/*.yaml for a quick smoke run.

variant_run_dirs = {}
train_script = repo_root / "train" / "train.py"

for name in VARIANTS:
    cfg_path = variant_config_paths[name]
    print("\n=== Training", name, "===")
    cmd = [sys.executable, str(train_script), "--config_path", str(cfg_path)]
    proc = subprocess.run(cmd, cwd=str(repo_root), capture_output=True, text=True)
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print(proc.stderr[-3000:])
        raise RuntimeError(f"Training failed for {name}")

    # Grab the newest run directory after each training run.
    run_dirs = sorted((repo_root / "runs").glob("run_*"), key=lambda p: p.stat().st_mtime)
    if not run_dirs:
        raise RuntimeError("No run directories found after training.")
    variant_run_dirs[name] = run_dirs[-1]

print("\nVariant run directories:")
for k, v in variant_run_dirs.items():
    print(f"  - {k}: {v}")

In [ ]:
from scripts.test_logic_vs_llama31 import run_benchmark

# Evaluate all four trained variants and keep frozen off-the-shelf Llama baseline enabled.
comparison_report = {
    "variants": {},
    "settings": {
        "datasets": ["cola", "proofwriter"],
        "split": "validation",
        "max_samples": 64,
        "batch_size": 8,
        "max_length": 256,
        "llama_model_name": "meta-llama/Llama-3.1-8B-Instruct",
        "skip_llama": False,
    },
}

for name in VARIANTS:
    run_dir = variant_run_dirs[name]
    print("\n=== Benchmark", name, "===")
    report = run_benchmark(
        run_dir=run_dir,
        datasets=comparison_report["settings"]["datasets"],
        split=comparison_report["settings"]["split"],
        max_samples=comparison_report["settings"]["max_samples"],
        batch_size=comparison_report["settings"]["batch_size"],
        max_length=comparison_report["settings"]["max_length"],
        llama_model_name=comparison_report["settings"]["llama_model_name"],
        skip_llama=comparison_report["settings"]["skip_llama"],
        output=repo_root / "runs" / f"logic_vs_llama31_{name}.json",
    )
    comparison_report["variants"][name] = report

summary_path = repo_root / "runs" / "four_way_comparison_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(comparison_report, f, indent=2)

print("\nSaved combined summary to", summary_path)
print(json.dumps(comparison_report, indent=2)[:6000])

## 9B  Visual Diagnostics For Four-Way Variants

Run these 2 code cells for Part 9 plots (after finishing 9A):

1. Build accuracy matrices + render grouped bars and delta heatmap: `#VSC-026e6384`
2. Optional logic-internals plot for one logic run: `#VSC-f8ba9ea3`

What this section shows:
- grouped bar charts for per-dataset accuracy
- project-minus-llama delta heatmap
- optional logic internals view (gate activations + routing map)

Dependency note: this section expects `comparison_report` in memory or `runs/four_way_comparison_summary.json` on disk from 9A.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Resolve repo root if this cell is run standalone.
if "repo_root" not in globals():
    _cwd = Path.cwd()
    _script_rel = Path("scripts") / "test_logic_vs_llama31.py"
    repo_root = _cwd
    if not (repo_root / _script_rel).exists():
        for candidate in [repo_root, *repo_root.parents]:
            if (candidate / _script_rel).exists():
                repo_root = candidate
                break

# Load comparison report from in-memory variable (preferred) or disk fallback.
if "comparison_report" in globals() and isinstance(comparison_report, dict):
    _cmp = comparison_report
else:
    _summary_path = repo_root / "runs" / "four_way_comparison_summary.json"
    if not _summary_path.exists():
        raise FileNotFoundError(
            f"Missing {_summary_path}. Run Part 9 training/benchmark cells first."
        )
    with open(_summary_path, "r", encoding="utf-8") as f:
        _cmp = json.load(f)

variants = list(_cmp.get("variants", {}).keys())
if not variants:
    raise RuntimeError("No variants found in comparison_report['variants']")

# Collect dataset keys across variants.
dataset_keys = sorted({
    ds
    for v in variants
    for ds in _cmp["variants"].get(v, {}).get("datasets", {}).keys()
})
if not dataset_keys:
    raise RuntimeError("No dataset metrics found in comparison report")

# Matrices: rows=variants, cols=datasets
proj_acc = np.full((len(variants), len(dataset_keys)), np.nan, dtype=float)
llama_acc = np.full((len(variants), len(dataset_keys)), np.nan, dtype=float)

for i, v in enumerate(variants):
    ds_block = _cmp["variants"].get(v, {}).get("datasets", {})
    for j, ds in enumerate(dataset_keys):
        d = ds_block.get(ds, {})
        proj_acc[i, j] = d.get("project_model", {}).get("accuracy", np.nan)
        llama_acc[i, j] = d.get("llama_3_1_8b", {}).get("accuracy", np.nan)

# 1) Grouped bar charts per dataset
num_ds = len(dataset_keys)
fig, axes = plt.subplots(1, num_ds, figsize=(6 * num_ds, 4), squeeze=False)
axes = axes[0]

for j, ds in enumerate(dataset_keys):
    ax = axes[j]
    x = np.arange(len(variants))
    width = 0.35

    ax.bar(x - width / 2, proj_acc[:, j], width=width, label="Project", color="#2f6db3")
    if np.isfinite(llama_acc[:, j]).any():
        ax.bar(x + width / 2, llama_acc[:, j], width=width, label="Llama 3.1 8B", color="#c44e52")

    ax.set_title(f"Accuracy on {ds}")
    ax.set_ylim(0.0, 1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(variants, rotation=20, ha="right")
    ax.set_ylabel("Accuracy")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="best")

plt.tight_layout()
plt.show()

# 2) Heatmap: project minus llama (positive is good)
delta = proj_acc - llama_acc
fig, ax = plt.subplots(figsize=(1.8 * len(dataset_keys) + 3, 0.7 * len(variants) + 2.5))
im = ax.imshow(delta, aspect="auto", cmap="RdBu_r", vmin=-0.5, vmax=0.5)

ax.set_title("Project - Llama accuracy delta")
ax.set_xticks(np.arange(len(dataset_keys)))
ax.set_xticklabels(dataset_keys)
ax.set_yticks(np.arange(len(variants)))
ax.set_yticklabels(variants)
ax.set_xlabel("Dataset")
ax.set_ylabel("Variant")

for i in range(len(variants)):
    for j in range(len(dataset_keys)):
        val = delta[i, j]
        txt = "nan" if not np.isfinite(val) else f"{val:+.3f}"
        ax.text(j, i, txt, ha="center", va="center", fontsize=9, color="black")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Accuracy delta")
plt.tight_layout()
plt.show()

# 3) Quick console summary for copy/paste
for i, v in enumerate(variants):
    parts = []
    for j, ds in enumerate(dataset_keys):
        pa = proj_acc[i, j]
        la = llama_acc[i, j]
        if np.isfinite(la):
            parts.append(f"{ds}: project={pa:.3f}, llama={la:.3f}, delta={pa-la:+.3f}")
        else:
            parts.append(f"{ds}: project={pa:.3f}, llama=NA")
    print(f"{v} -> " + " | ".join(parts))

In [ ]:
# Optional: reuse earlier-style internals visuals (gate activations + routing map)
# for one trained logic variant from Part 9.

import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer
from scripts.test_logic_vs_llama31 import _load_run_config, _build_project_model

# Resolve repo root if this cell is run standalone.
if "repo_root" not in globals():
    _cwd = Path.cwd()
    _script_rel = Path("scripts") / "test_logic_vs_llama31.py"
    repo_root = _cwd
    if not (repo_root / _script_rel).exists():
        for candidate in [repo_root, *repo_root.parents]:
            if (candidate / _script_rel).exists():
                repo_root = candidate
                break

# Choose a logic variant run dir from Part 9 outputs.
if "variant_run_dirs" in globals() and variant_run_dirs:
    _vrd = variant_run_dirs
else:
    # Fallback: infer from standard run-level outputs if available.
    _vrd = {}
    for name in ["logic_b", "logic_a"]:
        p = repo_root / "runs" / f"logic_vs_llama31_{name}.json"
        if p.exists():
            with open(p, "r", encoding="utf-8") as f:
                rep = json.load(f)
            rd = rep.get("run_dir")
            if rd:
                _vrd[name] = Path(rd)

if not _vrd:
    raise RuntimeError("No logic variant run found. Run Part 9 training/benchmark cells first.")

candidate_variant = "logic_b" if "logic_b" in _vrd else ("logic_a" if "logic_a" in _vrd else None)
if candidate_variant is None:
    raise RuntimeError("No logic variant run found (expected logic_a or logic_b)")

run_dir = Path(_vrd[candidate_variant])
if not run_dir.is_absolute():
    run_dir = (repo_root / run_dir).resolve()

cfg = _load_run_config(run_dir)
viz_model = _build_project_model(cfg)
state = torch.load(run_dir / "checkpoint.pt", map_location="cpu")
viz_model.load_state_dict(state, strict=False)

model_name = cfg["model"]["model_name"]
viz_tokenizer = AutoTokenizer.from_pretrained(model_name)
if viz_tokenizer.pad_token is None and viz_tokenizer.eos_token is not None:
    viz_tokenizer.pad_token = viz_tokenizer.eos_token

device_viz = torch.device("cuda" if torch.cuda.is_available() else "cpu")
viz_model = viz_model.to(device_viz).eval()

sample_text = [
    "All cats are animals. Felix is a cat.",
    "If it rains then streets are wet. Streets are not wet.",
]
enc = viz_tokenizer(sample_text, return_tensors="pt", padding=True, truncation=True, max_length=96)
input_ids = enc["input_ids"].to(device_viz)
attention_mask = enc["attention_mask"].to(device_viz)

with torch.no_grad():
    out = viz_model(input_ids=input_ids, attention_mask=attention_mask)

if not hasattr(out, "routing_history") or len(out.routing_history) == 0:
    print(f"Variant '{candidate_variant}' has no routing_history (likely baseline model).")
else:
    # Routing heatmap across layers (same spirit as earlier Section 6)
    route_mat = torch.stack([rw[0].mean(dim=0) for rw in out.routing_history]).cpu().numpy()
    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(route_mat, aspect="auto", cmap="viridis", interpolation="nearest")
    ax.set_title(f"{candidate_variant}: mean token-to-gate routing weights")
    ax.set_xlabel("Gate index")
    ax.set_ylabel("Layer")
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

    # Gate activation bars on first logic layer (same spirit as earlier Section 5)
    first_layer = viz_model.logic_stream.layers[0]
    bb_out = viz_model.backbone(input_ids[:1], attention_mask=attention_mask[:1])
    layer_hidden = list(bb_out.hidden_states[1:])
    token_logic = viz_model.logic_projection(layer_hidden[0])
    routing_w = first_layer.routing(token_logic, inference=True)
    gate_inputs = torch.einsum("bsg,bsl->bgl", routing_w, token_logic)
    gate_values = torch.sigmoid(first_layer.gate_scalar(gate_inputs).squeeze(-1))[0].cpu().numpy()

    fig, ax = plt.subplots(figsize=(10, 2.8))
    ax.bar(np.arange(len(gate_values)), gate_values, color="#3b7ddd", edgecolor="black", linewidth=0.4)
    ax.axhline(0.5, color="red", linestyle="--", linewidth=0.9)
    ax.set_title(f"{candidate_variant}: first-layer gate activations (pre-clip)")
    ax.set_xlabel("Gate index")
    ax.set_ylabel("Value")
    ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.show()

# Eval 4: No-Gate Parallel Stream Control (Attribution Test)

This eval adds a stronger control for attribution: keep a parallel stream + fusion path, but remove gate/routing logic.

Goal: test whether gains come from logic-specific computation versus simply adding extra trainable pathway capacity.

Run in order:
1. Build and write no-gate experiment configs
2. Train and benchmark no-gate variants (guarded; only runs if model support exists)

Recommended comparison set:
- Baseline A/B (Eval 3)
- No-Gate A/B (this eval)
- Logic A/B (Eval 3)

Primary interpretation:
- No-Gate close to Logic -> gains likely from extra pathway/capacity
- Logic clearly above No-Gate at similar parameter budget -> logic operations likely add value

In [ ]:
# Parameter accounting helper for attribution controls (baseline vs added parallel-stream capacity).
import torch.nn as nn

if "model" not in globals():
    raise RuntimeError("Run Eval 1 through model construction first so `model` exists.")

def _count_params(module: nn.Module, trainable_only: bool = False) -> int:
    if trainable_only:
        return sum(p.numel() for p in module.parameters() if p.requires_grad)
    return sum(p.numel() for p in module.parameters())

total_params = _count_params(model, trainable_only=False)
trainable_params = _count_params(model, trainable_only=True)

# For logic-style wrappers, estimate added capacity over a baseline-like wrapper
# (backbone + task head only) by summing non-baseline path modules.
added_modules = []
for name in ["logic_projection", "logic_stream", "fusion", "pre_head_norm"]:
    if hasattr(model, name):
        added_modules.append((name, getattr(model, name)))

added_total = sum(_count_params(m) for _, m in added_modules)
added_trainable = sum(_count_params(m, trainable_only=True) for _, m in added_modules)

print("== Parameter Report ==")
print(f"Model class            : {model.__class__.__name__}")
print(f"Total params           : {total_params:,}")
print(f"Trainable params       : {trainable_params:,}")

if added_modules:
    print(f"Added path params      : {added_total:,}  ({100*added_total/max(total_params,1):.2f}%)")
    print(f"Added path trainable   : {added_trainable:,}  ({100*added_trainable/max(trainable_params,1):.2f}% of trainable)")
    print("Added modules:", ", ".join(name for name, _ in added_modules))
    baseline_like_total = total_params - added_total
    print(f"Baseline-like estimate : {baseline_like_total:,}  (backbone + task head)")
else:
    print("No logic-path modules detected on `model`; this looks baseline-like.")

if hasattr(model, "logic_stream") and hasattr(model.logic_stream, "use_no_gate_stream") and model.logic_stream.use_no_gate_stream:
    print("No-gate stream enabled : True")
    print("No-gate update width   :", model.logic_stream.no_gate_update_hidden_dim)

# If a future no-gate implementation adds dedicated modules, list them explicitly too.
nogate_attrs = [
    "no_gate_stream",
    "no_gate_update",
    "parallel_update",
    "stream_update",
]
present_nogate = [name for name in nogate_attrs if hasattr(model, name)]
if present_nogate:
    nogate_total = sum(_count_params(getattr(model, name)) for name in present_nogate)
    print(f"No-gate-specific params: {nogate_total:,} via {present_nogate}")

In [ ]:
# Build no-gate variant configs (parallel stream retained, gates/routing disabled by config flags).
from pathlib import Path
from datetime import datetime
import copy
import os
import sys
import yaml

# Reuse repo_root/base_cfg from Eval 3 if present; else resolve lazily.
if "repo_root" not in globals():
    repo_root = Path.cwd()
    if not (repo_root / "train" / "train.py").exists():
        for candidate in [repo_root, *repo_root.parents]:
            if (candidate / "train" / "train.py").exists():
                repo_root = candidate
                break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

if "base_cfg" not in globals():
    from train.train import _load_config
    base_cfg = _load_config(repo_root / "configs" / "config.yaml")

# W&B setup for Eval 4 (safe when run standalone).
wandb_cfg = base_cfg.setdefault("wandb", {})
wandb_enabled = bool(wandb_cfg.get("enabled", False))

if wandb_enabled:
    wandb_key = None

    if "google.colab" in sys.modules:
        try:
            from google.colab import userdata
            wandb_key = userdata.get("WANDB_API_KEY") or userdata.get("WANDB_KEY")
        except Exception:
            wandb_key = None

    if not wandb_key:
        wandb_key = os.getenv("WANDB_API_KEY") or os.getenv("WANDB_KEY")

    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
        try:
            import wandb
            wandb.login(key=wandb_key, relogin=True)
            print("wandb login successful for Eval 4.")
        except Exception as e:
            print(f"wandb login failed ({type(e).__name__}: {e}). Disabling wandb for no-gate runs.")
            wandb_cfg["enabled"] = False
    else:
        print("No WANDB key found in Colab secrets or environment; disabling wandb for no-gate runs.")
        wandb_cfg["enabled"] = False
else:
    print("wandb disabled in base config for Eval 4.")

base_model_name = globals().get(
    "MODEL_NAME",
    base_cfg.get("model", {}).get("model_name", "meta-llama/Meta-Llama-3.1-8B"),
)

EXPERIMENT_CONFIG_DIR = repo_root / "configs" / "experiments"
EXPERIMENT_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
EVAL4_WANDB_GROUP = f"eval4_nogate_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print("Eval 4 wandb group:", EVAL4_WANDB_GROUP)

def _build_no_gate_config(base_cfg: dict, variant_name: str, model_name: str) -> dict:
    cfg = copy.deepcopy(base_cfg)
    cfg.setdefault("model", {})
    cfg.setdefault("train", {})
    cfg.setdefault("data", {})
    cfg.setdefault("wandb", {})

    # Keep the parallel stream path enabled, but request no-gate behavior.
    cfg["model"]["model_name"] = model_name
    cfg["model"]["use_logic_stream"] = True
    cfg["model"]["freeze_backbone"] = True
    cfg["model"]["use_no_gate_stream"] = True
    cfg["model"]["no_gate_update_type"] = "mlp"
    # Auto-match no-gate MLP width to approximate logic-layer parameter budget.
    cfg["model"]["no_gate_match_logic_params"] = True
    cfg["model"]["no_gate_param_match"] = True  # alias for compatibility
    cfg["model"]["no_gate_update_hidden_dim"] = None

    # Ensure generated configs carry resolved wandb enabled/disabled state.
    cfg["wandb"]["enabled"] = bool(base_cfg.get("wandb", {}).get("enabled", False))
    cfg["wandb"]["group"] = EVAL4_WANDB_GROUP
    cfg["wandb"]["run_name"] = f"eval4_{variant_name}"

    # Keep default stream dims if absent.
    cfg["model"]["logic_dim"] = int(cfg["model"].get("logic_dim", 256))
    cfg["model"]["num_gates"] = int(cfg["model"].get("num_gates", 32))
    cfg["model"]["routing_top_k"] = int(cfg["model"].get("routing_top_k", 2))
    cfg["model"]["fusion_hidden_dim"] = int(cfg["model"].get("fusion_hidden_dim", 1024))
    cfg["model"]["num_labels"] = int(cfg["model"].get("num_labels", 2))

    if variant_name == "nogate_a":
        cfg["model"]["lora"] = {"enabled": False}
    elif variant_name == "nogate_b":
        cfg["model"]["lora"] = {
            "enabled": True,
            "r": 8,
            "lora_alpha": 16,
            "dropout": 0.05,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        }
    else:
        raise ValueError(f"Unknown no-gate variant: {variant_name}")

    # Keep run budgets moderate unless user has already overridden.
    cfg["data"]["max_samples"] = int(cfg["data"].get("max_samples", 256) or 256)
    cfg["train"]["epochs"] = int(cfg["train"].get("epochs", 3))
    cfg["train"]["batch_size"] = int(cfg["train"].get("batch_size", 8))
    return cfg

NOGATE_VARIANTS = ["nogate_a", "nogate_b"]
nogate_config_paths = {}
for name in NOGATE_VARIANTS:
    cfg = _build_no_gate_config(base_cfg, name, model_name=base_model_name)
    out_path = EXPERIMENT_CONFIG_DIR / f"{name}.yaml"
    with open(out_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    nogate_config_paths[name] = out_path

print("Wrote no-gate configs:")
for k, v in nogate_config_paths.items():
    print(f"  - {k}: {v}")

In [ ]:
# Train + benchmark no-gate variants.
# This cell is guarded: it only runs if the codebase has explicit no-gate stream support.
from pathlib import Path
import json
import subprocess
import sys
from scripts.test_logic_vs_llama31 import run_benchmark

def _has_no_gate_support() -> bool:
    try:
        import inspect
        from logic.core.logic_llama_model import LogicLlamaModel
        sig = inspect.signature(LogicLlamaModel.__init__)
        return (
            "use_no_gate_stream" in sig.parameters
            or "no_gate_update_type" in sig.parameters
            or "no_gate_update_hidden_dim" in sig.parameters
        )
    except Exception:
        return False

if not _has_no_gate_support():
    raise RuntimeError(
        "No-gate stream config written, but model code does not yet expose no-gate controls. "
        "Implement no-gate stream support in logic/core first, then rerun this cell."
    )

train_script = repo_root / "train" / "train.py"
nogate_run_dirs = {}

for name in NOGATE_VARIANTS:
    cfg_path = nogate_config_paths[name]
    print("\n=== Training", name, "===")
    cmd = [sys.executable, str(train_script), "--config_path", str(cfg_path)]
    proc = subprocess.run(cmd, cwd=str(repo_root), capture_output=True, text=True)
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print(proc.stderr[-3000:])
        raise RuntimeError(f"Training failed for {name}")

    run_dirs = sorted((repo_root / "runs").glob("run_*"), key=lambda p: p.stat().st_mtime)
    if not run_dirs:
        raise RuntimeError("No run directories found after no-gate training.")
    nogate_run_dirs[name] = run_dirs[-1]

print("\nNo-gate run directories:")
for k, v in nogate_run_dirs.items():
    print(f"  - {k}: {v}")

nogate_report = {
    "variants": {},
    "settings": {
        "datasets": ["cola", "proofwriter"],
        "split": "validation",
        "max_samples": 64,
        "batch_size": 8,
        "max_length": 256,
        "llama_model_name": "meta-llama/Llama-3.1-8B-Instruct",
        "skip_llama": False,
    },
}

for name in NOGATE_VARIANTS:
    print("\n=== Benchmark", name, "===")
    report = run_benchmark(
        run_dir=nogate_run_dirs[name],
        datasets=nogate_report["settings"]["datasets"],
        split=nogate_report["settings"]["split"],
        max_samples=nogate_report["settings"]["max_samples"],
        batch_size=nogate_report["settings"]["batch_size"],
        max_length=nogate_report["settings"]["max_length"],
        llama_model_name=nogate_report["settings"]["llama_model_name"],
        skip_llama=nogate_report["settings"]["skip_llama"],
        output=repo_root / "runs" / f"logic_vs_llama31_{name}.json",
    )
    nogate_report["variants"][name] = report

summary_path = repo_root / "runs" / "nogate_comparison_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(nogate_report, f, indent=2)

print("\nSaved no-gate summary to", summary_path)
print(json.dumps(nogate_report, indent=2)[:6000])

# Eval 5: LoRA No-Gate vs LoRA Soft Gates vs LoRA STE Gates (20 Epochs, Warmup -> Cyclic LR)

This eval runs a focused comparison between three LoRA-enabled variants:
- Variant 1: no-gate parallel stream (use_no_gate_stream = true)
- Variant 2: gate_mode = soft
- Variant 3: gate_mode = ste_binary

Training policy (matched across all three):
- epochs = 20
- scheduler = linear warmup, then cyclic learning rate

Run in order:
1. Build and write Eval 5 configs
2. Train/evaluate all three variants (no-gate first), benchmark against Llama

In [ ]:
# Build Eval 5 configs: LoRA + no-gate stream vs LoRA + soft gates vs LoRA + STE gates.
from pathlib import Path
from datetime import datetime
import copy
import os
import sys
import yaml

if "repo_root" not in globals():
    repo_root = Path.cwd()
    if not (repo_root / "train" / "train.py").exists():
        for candidate in [repo_root, *repo_root.parents]:
            if (candidate / "train" / "train.py").exists():
                repo_root = candidate
                break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

if "base_cfg" not in globals():
    from train.train import _load_config
    base_cfg = _load_config(repo_root / "configs" / "config.yaml")

# Reuse WANDB flow from prior evals for standalone execution.
wandb_cfg = base_cfg.setdefault("wandb", {})
if bool(wandb_cfg.get("enabled", False)):
    wandb_key = None
    if "google.colab" in sys.modules:
        try:
            from google.colab import userdata
            wandb_key = userdata.get("WANDB_API_KEY") or userdata.get("WANDB_KEY")
        except Exception:
            wandb_key = None
    if not wandb_key:
        wandb_key = os.getenv("WANDB_API_KEY") or os.getenv("WANDB_KEY")
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
        try:
            import wandb
            wandb.login(key=wandb_key, relogin=True)
            print("wandb login successful for Eval 5.")
        except Exception as e:
            print(f"wandb login failed ({type(e).__name__}: {e}). Disabling wandb for Eval 5 runs.")
            wandb_cfg["enabled"] = False
    else:
        print("No WANDB key found; disabling wandb for Eval 5 runs.")
        wandb_cfg["enabled"] = False
else:
    print("wandb disabled in base config for Eval 5.")

base_model_name = globals().get(
    "MODEL_NAME",
    base_cfg.get("model", {}).get("model_name", "meta-llama/Meta-Llama-3.1-8B"),
)

EXPERIMENT_CONFIG_DIR = repo_root / "configs" / "experiments"
EXPERIMENT_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
EVAL5_WANDB_GROUP = f"eval5_lora_gate_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print("Eval 5 wandb group:", EVAL5_WANDB_GROUP)

def _build_eval5_config(base_cfg: dict, variant_name: str, model_name: str) -> dict:
    cfg = copy.deepcopy(base_cfg)
    cfg.setdefault("model", {})
    cfg.setdefault("train", {})
    cfg.setdefault("data", {})
    cfg.setdefault("wandb", {})

    # Core setup: logic path + LoRA on backbone.
    cfg["model"]["model_name"] = model_name
    cfg["model"]["use_logic_stream"] = True
    cfg["model"]["freeze_backbone"] = True
    cfg["model"]["lora"] = {
        "enabled": True,
        "r": 8,
        "lora_alpha": 16,
        "dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    }

    # Variant split for this ablation.
    cfg["model"]["use_no_gate_stream"] = False
    cfg["model"]["no_gate_match_logic_params"] = False
    cfg["model"]["no_gate_update_hidden_dim"] = None
    if variant_name == "lora_nogate":
        cfg["model"]["use_no_gate_stream"] = True
        cfg["model"]["no_gate_match_logic_params"] = True
        cfg["model"]["no_gate_param_match"] = True
        cfg["model"]["no_gate_update_type"] = "mlp"
        cfg["model"]["gate_mode"] = "soft"  # ignored in no-gate mode
    elif variant_name == "lora_soft":
        cfg["model"]["gate_mode"] = "soft"
    elif variant_name == "lora_ste":
        cfg["model"]["gate_mode"] = "ste_binary"
    else:
        raise ValueError(f"Unknown Eval 5 variant: {variant_name}")

    # Requested budget: 20 epochs + warmup -> cyclic scheduler.
    cfg["train"]["epochs"] = 20
    cfg["train"]["scheduler"] = {
        "type": "warmup_cyclic",
        "base_lr": float(cfg["train"].get("learning_rate", 2e-4)) * 0.25,
        "max_lr": float(cfg["train"].get("learning_rate", 2e-4)),
        "step_size_up": int(cfg["train"].get("scheduler", {}).get("step_size_up", 200)),
        "step_size_down": int(cfg["train"].get("scheduler", {}).get("step_size_down", 200)),
        "mode": "triangular2",
        "warmup_start_factor": 0.1,
    }

    # Keep sane quick defaults unless already set lower by user.
    cfg["data"]["max_samples"] = int(cfg["data"].get("max_samples", 256) or 256)
    cfg["train"]["batch_size"] = int(cfg["train"].get("batch_size", 8))

    cfg["wandb"]["enabled"] = bool(base_cfg.get("wandb", {}).get("enabled", False))
    cfg["wandb"]["group"] = EVAL5_WANDB_GROUP
    cfg["wandb"]["run_name"] = f"eval5_{variant_name}"
    return cfg

# Run no-gate first, then gated variants.
EVAL5_VARIANTS = ["lora_nogate", "lora_soft", "lora_ste"]
eval5_config_paths = {}
for name in EVAL5_VARIANTS:
    cfg = _build_eval5_config(base_cfg, name, model_name=base_model_name)
    out_path = EXPERIMENT_CONFIG_DIR / f"{name}.yaml"
    with open(out_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    eval5_config_paths[name] = out_path

print("Wrote Eval 5 configs:")
for k, v in eval5_config_paths.items():
    print(f"  - {k}: {v}")

In [ ]:
# Train + benchmark Eval 5 variants.
from pathlib import Path
import json
import subprocess
import sys
from scripts.test_logic_vs_llama31 import run_benchmark

train_script = repo_root / "train" / "train.py"
eval5_run_dirs = {}

for name in EVAL5_VARIANTS:
    cfg_path = eval5_config_paths[name]
    print("\n=== Training", name, "===")
    cmd = [sys.executable, str(train_script), "--config_path", str(cfg_path)]
    proc = subprocess.run(cmd, cwd=str(repo_root), capture_output=True, text=True)
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print(proc.stderr[-3000:])
        raise RuntimeError(f"Training failed for {name}")

    run_dirs = sorted((repo_root / "runs").glob("run_*"), key=lambda p: p.stat().st_mtime)
    if not run_dirs:
        raise RuntimeError("No run directories found after Eval 5 training.")
    eval5_run_dirs[name] = run_dirs[-1]

print("\nEval 5 run directories:")
for k, v in eval5_run_dirs.items():
    print(f"  - {k}: {v}")

eval5_report = {
    "variants": {},
    "settings": {
        "datasets": ["cola", "proofwriter"],
        "split": "validation",
        "max_samples": 64,
        "batch_size": 8,
        "max_length": 256,
        "llama_model_name": "meta-llama/Llama-3.1-8B-Instruct",
        "skip_llama": False,
    },
}

for name in EVAL5_VARIANTS:
    print("\n=== Benchmark", name, "===")
    report = run_benchmark(
        run_dir=eval5_run_dirs[name],
        datasets=eval5_report["settings"]["datasets"],
        split=eval5_report["settings"]["split"],
        max_samples=eval5_report["settings"]["max_samples"],
        batch_size=eval5_report["settings"]["batch_size"],
        max_length=eval5_report["settings"]["max_length"],
        llama_model_name=eval5_report["settings"]["llama_model_name"],
        skip_llama=eval5_report["settings"]["skip_llama"],
        output=repo_root / "runs" / f"logic_vs_llama31_{name}.json",
    )
    eval5_report["variants"][name] = report

summary_path = repo_root / "runs" / "lora_soft_vs_ste_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(eval5_report, f, indent=2)

print("\nSaved Eval 5 summary to", summary_path)
print(json.dumps(eval5_report, indent=2)[:6000])

In [ ]:
# Eval 6: Test several inference-time top-k values vs a soft-routing test baseline.
from pathlib import Path
import copy
import json
import torch
from transformers import AutoTokenizer
from scripts.test_logic_vs_llama31 import (
    _load_run_config,
    _build_project_model,
    _require_dict,
    _require_keys,
    run_benchmark_with_model,
)

if "repo_root" not in globals():
    repo_root = Path.cwd()
    if not (repo_root / "train" / "train.py").exists():
        for candidate in [repo_root, *repo_root.parents]:
            if (candidate / "train" / "train.py").exists():
                repo_root = candidate
                break

# Prefer the Eval 5 soft-gate run as the base checkpoint.
if "eval5_run_dirs" in globals() and isinstance(eval5_run_dirs, dict) and "lora_soft" in eval5_run_dirs:
    eval6_base_run_dir = Path(eval5_run_dirs["lora_soft"])
else:
    run_dirs = sorted((repo_root / "runs").glob("run_*"), key=lambda p: p.stat().st_mtime)
    if not run_dirs:
        raise RuntimeError("No run directories found. Train a model first.")
    eval6_base_run_dir = run_dirs[-1]

print("Eval 6 base run:", eval6_base_run_dir)

run_cfg = _load_run_config(eval6_base_run_dir)
model_cfg = _require_dict(run_cfg, "model")
_require_keys(model_cfg, "model", ["model_name", "num_gates", "routing_top_k"])
num_gates = int(model_cfg["num_gates"])
raw_topk_values = [1, 2, 4, 8]
eval6_topk_values = sorted({min(k, num_gates) for k in raw_topk_values})

# soft_test uses k=num_gates, which preserves full soft routing at inference.
eval6_variants = {"soft_test": num_gates}
for k in eval6_topk_values:
    eval6_variants[f"topk_{k}"] = k

eval6_settings = {
    "datasets": ["cola", "proofwriter"],
    "split": "validation",
    "max_samples": 64,
    "batch_size": 8,
    "max_length": 256,
    "seed": 42,
    "llama_model_name": "meta-llama/Llama-3.1-8B-Instruct",
    "skip_llama": True,
}

if "eval5_report" in globals() and isinstance(eval5_report, dict) and "settings" in eval5_report:
    inherited = eval5_report["settings"]
    for key in [
        "datasets",
        "split",
        "max_samples",
        "batch_size",
        "max_length",
        "llama_model_name",
        "skip_llama",
    ]:
        if key in inherited:
            eval6_settings[key] = inherited[key]

checkpoint_path = eval6_base_run_dir / "checkpoint.pt"
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Missing checkpoint: {checkpoint_path}")
checkpoint_state = torch.load(checkpoint_path, map_location="cpu")

project_tokenizer = AutoTokenizer.from_pretrained(model_cfg["model_name"])
if project_tokenizer.pad_token is None and project_tokenizer.eos_token is not None:
    project_tokenizer.pad_token = project_tokenizer.eos_token

eval6_report = {
    "base_run_dir": str(eval6_base_run_dir),
    "settings": eval6_settings,
    "variants": {},
}

for variant_name, k_value in eval6_variants.items():
    print(f"\n=== Eval 6 benchmark: {variant_name} (routing_top_k={k_value}) ===")
    variant_cfg = copy.deepcopy(run_cfg)
    variant_cfg["model"]["routing_top_k"] = int(k_value)

    project_model = _build_project_model(variant_cfg)
    project_model.load_state_dict(checkpoint_state, strict=False)

    output_path = repo_root / "runs" / f"logic_vs_llama31_eval6_{variant_name}.json"
    report = run_benchmark_with_model(
        project_model=project_model,
        project_tokenizer=project_tokenizer,
        datasets=eval6_settings["datasets"],
        split=eval6_settings["split"],
        max_samples=int(eval6_settings["max_samples"]),
        batch_size=int(eval6_settings["batch_size"]),
        max_length=int(eval6_settings["max_length"]),
        seed=int(eval6_settings["seed"]),
        llama_model_name=eval6_settings["llama_model_name"],
        skip_llama=bool(eval6_settings["skip_llama"]),
        output=output_path,
        project_model_name=f"eval6_{variant_name}",
    )
    eval6_report["variants"][variant_name] = report

summary_path = repo_root / "runs" / "eval6_topk_vs_soft_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(eval6_report, f, indent=2)

print("\nSaved Eval 6 summary to", summary_path)
print("\nQuick accuracy snapshot:")
for variant_name, variant_report in eval6_report["variants"].items():
    print(f"  [{variant_name}]")
    for ds_name, ds_report in variant_report["datasets"].items():
        acc = ds_report["project_model"]["accuracy"]
        print(f"    - {ds_name}: {acc:.4f}")